# Potential Talents - Candidate Ranking
## Notebook 02 - Stage 1 NLPL Model 40 Word2Vec ranking

Uses **NLPL model 40 only** (English CoNLL17 Word2Vec, 100D). Put `40.zip` in the project root or set `NLPL_MODEL40_ZIP`.

In [1]:
import re, numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score
source=pd.read_csv('potential-talents.csv')
labels=pd.read_csv('unique-job-titles-labelled.csv').sort_values('representative_id').reset_index(drop=True)
assert len(source)==104 and source['job_title'].nunique()==52 and len(labels)==52
DIRECT=re.compile(r'\bhuman\s+resources\b|\bhr\b|\bhris\b|\bchro\b',re.I)
ADJ=re.compile(r'\bpeople\s+development\b|\btalent\s+(?:acquisition|management)\b|\bstaffing\b|\brecruit(?:er|ing|ment)\b|\bbenefits\b|\bcompensation\b',re.I)
SOLICIT=re.compile(r'\b(?:is|are)\s+seeking\b.*\b(?:professionals?|candidates?|applicants?)\b',re.I)
def H(t):
    t=str(t)
    if SOLICIT.search(t): return 0.0
    if DIRECT.search(t): return 1.0
    if ADJ.search(t): return 0.5
    return 0.0
def I(t,q):
    t,q=str(t).lower(),str(q).lower(); target='aspiring' if 'aspiring' in q else 'seeking'; other='seeking' if target=='aspiring' else 'aspiring'
    return 1.0 if target in t else (0.5 if other in t else 0.0)
def rule_score(t,q): return H(t)*(0.70+0.30*I(t,q))
def rule_level(t):
    h=H(t)
    if h==0:return 0
    if h<1:return 1
    tl=str(t).lower(); return 3 if ('aspiring' in tl or 'seeking' in tl) else 2
freq=source['job_title'].value_counts(); human=labels['manual_relevance_grade'].to_numpy(int); rule=labels['job_title'].map(rule_level).to_numpy(int)
print({'source_rows':len(source),'unique_titles':source['job_title'].nunique(),'repeated_unique_titles':int((freq>1).sum()),'max_frequency':int(freq.max()),'exact_agreement':round(float(np.mean(human==rule)),3),'kappa':round(float(cohen_kappa_score(human,rule,weights='quadratic')),3),'spearman':round(float(spearmanr(human,rule).statistic),3)})
import os, zipfile
from pathlib import Path
from sklearn.metrics import ndcg_score
from sklearn.metrics.pairwise import cosine_similarity
QUERIES=('aspiring human resources','seeking human resources')
_PHONE=re.compile(r'\(?\d{3}\)?\s*[- ]\s*\d{3}\s*[- ]\s*\d{4}'); _YEAR=re.compile(r'\b(?:19|20)\d{2}\b'); _TOKEN=re.compile(r"[A-Za-z]+(?:['’][A-Za-z]+)?|\d+")
def audit_tokens(text):
    text=_YEAR.sub(' ',_PHONE.sub(' ',str(text).replace('’',"'"))); toks=_TOKEN.findall(text)
    return [t for t in toks if t.lower()!='at' and not(len(t)==1 and t.isupper())]
texts=list(labels['job_title'].astype(str))+list(QUERIES); raw=[t for s in texts for t in audit_tokens(s)]; needed=set()
for t in raw:
    needed|={t,t.lower()}
    if t.lower().endswith("'s"): needed|={t[:-2],t[:-2].lower()}
zip_path=Path(os.environ.get('NLPL_MODEL40_ZIP','40.zip')); vectors={}
with zipfile.ZipFile(zip_path) as z:
    member=[n for n in z.namelist() if n.endswith('model.txt')][0]
    with z.open(member) as fh:
        vocab,dim=map(int,fh.readline().decode().split()[:2]); assert dim==100
        for line in fh:
            pos=line.find(b' ')
            if pos<=0: continue
            try: word=line[:pos].decode('utf-8')
            except UnicodeDecodeError: continue
            if word in needed:
                v=np.fromstring(line[pos+1:].decode('ascii','ignore'),sep=' ',dtype=np.float32)
                if v.size==dim:vectors[word]=v
def resolve(t):
    for x in (t,t.lower()):
        if x in vectors:return x
    if t.lower().endswith("'s"):
        for x in (t[:-2],t[:-2].lower()):
            if x in vectors:return x
def meanvec(s):
    rs=[resolve(t) for t in audit_tokens(s)]; rs=[r for r in rs if r]; return np.mean(np.vstack([vectors[r] for r in rs]),axis=0)
def unit(v): return v/np.linalg.norm(v)
X=np.vstack([meanvec(t) for t in labels['job_title']]); ids=labels['representative_id'].to_numpy(int); base={}
for q in QUERIES:
    qv=unit(meanvec(q)); sims=cosine_similarity(X,qv.reshape(1,-1)).ravel(); order=np.lexsort((ids,-sims)); rel=labels['job_title'].map(lambda t:rule_score(t,q)).to_numpy(float); n=float(ndcg_score(rel.reshape(1,-1),sims.reshape(1,-1),k=10)); base[q]={'q':qv,'sims':sims,'order':order,'top10':set(order[:10]),'ndcg':n}; print(q,'NDCG@10',round(n,3)); print(labels.iloc[order[:10]][['representative_id','job_title']].assign(cosine=sims[order[:10]]).to_string(index=False))
print('coverage',sum(resolve(t) is not None for t in raw),'/',len(raw),'OOV unique',len({t for t in raw if resolve(t) is None}))

### Verified live-run checkpoints

| Query | NDCG@10 |
|---|---:|
| aspiring human resources | **0.948** |
| seeking human resources | **0.935** |

All 374 meaningful title/query token occurrences resolved to vectors; unique OOV count = **0**. Top 3 aspiring IDs: **6, 3, 73**. Top 3 seeking IDs: **28, 99, 73**.